# Preprocessing Pipeline (Sections 2-4)
# Integrated Multi-Dataset Deepfake Detection

## Overview
This notebook runs the **one-time preprocessing pipeline**: dataset loading,
face extraction, and feature extraction.

Run this notebook **once** on the full dataset.  It saves all outputs needed
for training in `train_sections5_end.ipynb`.

### Pipeline Stages
1. **Section 2** - Load dataset metadata (FF++, Celeb-DF, DFDC, HiDF)
2. **Section 3** - Face detection & extraction (OpenCV DNN)
3. **Section 4** - Multi-domain feature extraction (Spatial, Frequency, Semantic)
4. **Manifest** - Save sample manifest for Notebook B

### Outputs
- `preprocessed_faces_integrated/` - Extracted face images
- `extracted_features_integrated/` - Feature vectors (.npy)
- `preprocessing_manifest.csv` - Sample manifest for training notebook
- `pipeline_tracking.csv` - Pipeline stage counts

## Section 1: Setup & Configuration

In [1]:
!pip install --user timm imageio imageio-ffmpeg av

  Using cached timm-1.0.25-py3-none-any.whl.metadata (38 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached av-16.1.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (4.6 kB)
Using cached timm-1.0.25-py3-none-any.whl (2.6 MB)
Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl (29.5 MB)
Using cached av-16.1.0-cp313-cp313-manylinux_2_28_x86_64.whl (40.9 MB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import json
import numpy as np
import pandas as pd
import random
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import urllib.request

# Image processing
import cv2
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms

# Models
import site
import sys

sys.path.append(site.getusersitepackages())

import timm
# Face detection via OpenCV DNN (facenet-pytorch can't build on Python 3.13/SWAN)

# Analysis and visualization
from scipy.stats import skew, kurtosis
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# ============================================================================
# SET RANDOM SEED
# ============================================================================

def set_seed(seed=42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"{'='*70}")
print("ENVIRONMENT SETUP")
print(f"{'='*70}")
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")
print(f"{'='*70}")

ENVIRONMENT SETUP
Using device: cuda
PyTorch Version: 2.10.0


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Central configuration for the integrated training pipeline."""
    
    # ==========================================================================
    # DATA PERCENTAGE - Controls how much data to use from each dataset
    # ==========================================================================
    # Global default (used if per-dataset value not specified)
    # DATA_PERCENTAGE = 0.01  # 1% default
    
    # Per-dataset percentages (0.01 to 1.0)
    FFPP_PERCENTAGE = 1       # FaceForensics++ percentage
    CELEBDF_PERCENTAGE = 1  # Celeb-DF percentage
    DFDC_PERCENTAGE = 1      # DFDC percentage
    HIDF_PERCENTAGE = 1      # HiDF percentage
    
    # ==========================================================================
    # Dataset Paths (SWAN CERN Cloud)
    # ==========================================================================
    BASE_DIR = Path('.')  # Current directory (same as notebook)
    
    # Individual dataset paths (extra folder layer from zip extraction)
    FFPP_DIR = BASE_DIR / 'FF' / 'FaceForensics++'
    CELEBDF_DIR = BASE_DIR / 'CelebDF' / 'CelebDF'
    DFDC_DIR = BASE_DIR / 'DDFC' / 'deepfake-detection-challenge'
    HIDF_DIR = BASE_DIR / 'HiDF' / 'HiDF'
    
    # Output directories
    PREPROCESSED_DIR = BASE_DIR / 'preprocessed_faces_integrated'
    FEATURES_DIR = BASE_DIR / 'extracted_features_integrated'
    CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'
    RESULTS_DIR = BASE_DIR / 'results'
    INTEGRATED_METADATA_PATH = BASE_DIR / 'integrated_nocuda.csv'
    
    # Data settings
    FRAMES_PER_VIDEO = 20
    FACE_SIZE = 299
    TRAIN_RATIO = 0.8
    
    # Feature dimensions
    SPATIAL_DIM = 2048   # Xception output
    FREQ_DIM = 128         # Azimuthal power spectrum (128-dim)
    SEMANTIC_DIM = 768   # DINOv2 ViT-B/14 output
    
    # Training settings
    BATCH_SIZE = 128
    LEARNING_RATE = 1e-4
    EPOCHS = 20
    HIDDEN_DIM1 = 512
    HIDDEN_DIM2 = 256
    DROPOUT = 0.5
    FUSION_DIM = 512
    
    # Early stopping
    PATIENCE = 5
    MIN_DELTA = 0.001

config = Config()

# Create directory structure
def create_directories():
    """Create all necessary directories."""
    dirs = [
        config.PREPROCESSED_DIR / 'real',
        config.PREPROCESSED_DIR / 'fake',
        config.FEATURES_DIR / 'spatial',
        config.FEATURES_DIR / 'frequency',
        config.FEATURES_DIR / 'semantic',
        config.CHECKPOINTS_DIR,
        config.RESULTS_DIR
    ]
    for dir_path in dirs:
        dir_path.mkdir(parents=True, exist_ok=True)
    print("✓ Directory structure created")

create_directories()

# Display configuration
print(f"\n{'='*70}")
print("PROJECT CONFIGURATION")
print(f"{'='*70}")
print(f"Data Percentages:")
print(f"  - FaceForensics++: {config.FFPP_PERCENTAGE * 100:.1f}%")
print(f"  - Celeb-DF: {config.CELEBDF_PERCENTAGE * 100:.1f}%")
print(f"  - DFDC: {config.DFDC_PERCENTAGE * 100:.1f}%")
print(f"  - HiDF: {config.HIDF_PERCENTAGE * 100:.1f}%")
print(f"Base Directory: {config.BASE_DIR.absolute()}")
print(f"Frames per video: {config.FRAMES_PER_VIDEO}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Learning rate: {config.LEARNING_RATE}")
print(f"Epochs: {config.EPOCHS}")
print(f"Feature dimensions: Spatial={config.SPATIAL_DIM}, Freq={config.FREQ_DIM}, Semantic={config.SEMANTIC_DIM}")
print(f"{'='*70}")

## Section 2: Dataset Loaders

Load and parse metadata from all four datasets.

In [4]:
# ============================================================================
# FACEFORENSICS++ DATASET LOADER
# ============================================================================

def load_ffpp_dataset(data_percentage=None):
    """
    Load FaceForensics++ dataset metadata.
    Uses CSV files from datasets/FaceForensics++/csv/
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use FFPP-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.FFPP_PERCENTAGE
    csv_dir = config.FFPP_DIR / 'csv'
    
    if not csv_dir.exists():
        print(f"⚠ FaceForensics++ CSV directory not found: {csv_dir}")
        return pd.DataFrame()
    
    data = []
    
    # Load original (real) videos
    original_csv = csv_dir / 'original.csv'
    if original_csv.exists():
        df_orig = pd.read_csv(original_csv)
        # Sample the required percentage
        n_samples = max(1, int(len(df_orig) * data_percentage))
        df_orig_sampled = df_orig.sample(n=n_samples, random_state=42)
        
        for _, row in df_orig_sampled.iterrows():
            file_path = row['File Path']
            video_name = Path(file_path).stem
            full_path = config.FFPP_DIR / file_path
            if full_path.exists():
                data.append({
                    'video_id': f'FFPP_original_{video_name}',
                    'label': 'real',
                    'path': str(full_path),
                    'manipulation': 'Original',
                    'dataset': 'FaceForensics++'
                })
    
    # Load fake videos from different manipulation methods
    manipulation_csvs = ['Deepfakes.csv', 'Face2Face.csv', 'FaceSwap.csv', 
                         'NeuralTextures.csv', 'FaceShifter.csv']
    
    for csv_name in manipulation_csvs:
        csv_path = csv_dir / csv_name
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            manipulation = csv_name.replace('.csv', '')
            
            # Sample the required percentage
            n_samples = max(1, int(len(df) * data_percentage))
            df_sampled = df.sample(n=n_samples, random_state=42)
            
            for _, row in df_sampled.iterrows():
                file_path = row['File Path']
                video_name = Path(file_path).stem
                full_path = config.FFPP_DIR / file_path
                if full_path.exists():
                    data.append({
                        'video_id': f'FFPP_{manipulation}_{video_name}',
                        'label': 'fake',
                        'path': str(full_path),
                        'manipulation': manipulation,
                        'dataset': 'FaceForensics++'
                    })
    
    # Also check FF++_Metadata.csv for DeepFakeDetection videos
    metadata_csv = csv_dir / 'FF++_Metadata.csv'
    if metadata_csv.exists():
        df_meta = pd.read_csv(metadata_csv)
        # Filter for DeepFakeDetection
        df_dfd = df_meta[df_meta['File Path'].str.contains('DeepFakeDetection', na=False)]
        
        n_samples = max(1, int(len(df_dfd) * data_percentage))
        df_dfd_sampled = df_dfd.sample(n=min(n_samples, len(df_dfd)), random_state=42)
        
        for _, row in df_dfd_sampled.iterrows():
            file_path = row['File Path']
            video_name = Path(file_path).stem
            full_path = config.FFPP_DIR / file_path
            label = 'fake' if row['Label'] == 'FAKE' else 'real'
            if full_path.exists():
                data.append({
                    'video_id': f'FFPP_DeepFakeDetection_{video_name}',
                    'label': label,
                    'path': str(full_path),
                    'manipulation': 'DeepFakeDetection',
                    'dataset': 'FaceForensics++'
                })
    
    df_result = pd.DataFrame(data)
    print(f"✓ FaceForensics++: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [5]:
# ============================================================================
# CELEB-DF DATASET LOADER
# ============================================================================

def load_celebdf_dataset(data_percentage=None):
    """
    Load Celeb-DF dataset metadata.
    Uses List_of_testing_videos.txt for ground truth labels.
    
    Format: <label> <path>
    - Label 1 = Real
    - Label 0 = Fake (Celeb-synthesis)
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use Celeb-DF-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.CELEBDF_PERCENTAGE
    list_file = config.CELEBDF_DIR / 'List_of_testing_videos.txt'
    
    if not list_file.exists():
        print(f"⚠ Celeb-DF list file not found: {list_file}")
        return pd.DataFrame()
    
    data = []
    
    with open(list_file, 'r') as f:
        lines = f.readlines()
    
    # Parse all entries
    all_entries = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) >= 2:
            file_label = int(parts[0])
            file_path = parts[1]
            all_entries.append((file_label, file_path))
    
    # Sample the required percentage
    n_samples = max(1, int(len(all_entries) * data_percentage))
    random.seed(42)
    sampled_entries = random.sample(all_entries, min(n_samples, len(all_entries)))
    
    for file_label, file_path in sampled_entries:
        video_name = Path(file_path).stem
        folder_name = Path(file_path).parent.name
        full_path = config.CELEBDF_DIR / file_path
        
        # Label 1 = Real, Label 0 = Fake
        label = 'real' if file_label == 1 else 'fake'
        
        # Determine manipulation type based on folder
        if 'synthesis' in folder_name.lower():
            manipulation = 'Celeb-synthesis'
        elif 'real' in folder_name.lower():
            manipulation = 'Original'
        else:
            manipulation = folder_name
        
        if full_path.exists():
            data.append({
                'video_id': f'CelebDF_{folder_name}_{video_name}',
                'label': label,
                'path': str(full_path),
                'manipulation': manipulation,
                'dataset': 'Celeb-DF'
            })
    
    df_result = pd.DataFrame(data)
    print(f"✓ Celeb-DF: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [6]:
# ============================================================================
# DFDC (DEEPFAKE DETECTION CHALLENGE) DATASET LOADER
# ============================================================================

def load_dfdc_dataset(data_percentage=None):
    """
    Load DFDC dataset metadata.
    Uses metadata.json from train_sample_videos directory.
    
    Format: {"filename.mp4": {"label": "REAL"/"FAKE", "split": "train", "original": ...}}
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset
    """
    # Use DFDC-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.DFDC_PERCENTAGE
    videos_dir = config.DFDC_DIR / 'train_sample_videos'
    metadata_file = videos_dir / 'metadata.json'
    
    if not metadata_file.exists():
        print(f"⚠ DFDC metadata file not found: {metadata_file}")
        return pd.DataFrame()
    
    data = []
    
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    # Get all entries
    all_entries = list(metadata.items())
    
    # Sample the required percentage
    n_samples = max(1, int(len(all_entries) * data_percentage))
    random.seed(42)
    sampled_entries = random.sample(all_entries, min(n_samples, len(all_entries)))
    
    for filename, info in sampled_entries:
        video_path = videos_dir / filename
        video_name = Path(filename).stem
        label = 'real' if info['label'] == 'REAL' else 'fake'
        
        # Determine manipulation type
        if label == 'fake':
            manipulation = 'DFDC_DeepFake'
        else:
            manipulation = 'Original'
        
        if video_path.exists():
            data.append({
                'video_id': f'DFDC_{video_name}',
                'label': label,
                'path': str(video_path),
                'manipulation': manipulation,
                'dataset': 'DFDC'
            })
    
    df_result = pd.DataFrame(data)
    print(f"✓ DFDC: Loaded {len(df_result)} videos ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
    
    return df_result

In [7]:
# ============================================================================
# HiDF DATASET LOADER
# ============================================================================

def load_hidf_dataset(data_percentage=None):
    """
    Load HiDF dataset metadata using FOLDER-BASED LABELING.
    
    Labeling based on folder names:
    - Files in Fake-vid, Fake-img folders → label = "fake"
    - Files in Real-vid, Real-img folders → label = "real"
    
    This approach directly scans the folders instead of relying on metadata.csv,
    which ensures accurate labeling based on actual file locations.
    
    Directories:
    - Real-vid: Real videos (.mp4)
    - Real-img: Real images (.jpg, .png) - may not exist yet
    - Fake-vid: Fake videos (.mp4)
    - Fake-img: Fake images (.jpg, .png)
    
    Returns:
        DataFrame with columns: video_id, label, path, manipulation, dataset, media_type
    """
    # Use HiDF-specific percentage, or fallback to provided/global
    data_percentage = data_percentage or config.HIDF_PERCENTAGE
    
    data = []
    
    # Define folder configurations: (folder_name, label, media_type, manipulation)
    folder_configs = [
        ('Fake-vid', 'fake', 'vid', 'HiDF_FaceSwap'),
        ('Fake-img', 'fake', 'img', 'HiDF_FaceSwap'),
        ('Real-vid', 'real', 'vid', 'Original'),
        ('Real-img', 'real', 'img', 'Original'),
    ]
    
    for folder_name, label, media_type, manipulation in folder_configs:
        folder_path = config.HIDF_DIR / folder_name
        
        if not folder_path.exists():
            print(f"  ⚠ HiDF folder not found: {folder_name}")
            continue
        
        # Define file extensions based on media type
        if media_type == 'vid':
            extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv']
        else:
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        
        # Collect all files from this folder
        all_files = []
        for ext in extensions:
            all_files.extend(list(folder_path.glob(ext)))
        
        if not all_files:
            print(f"  ⚠ No files found in {folder_name}")
            continue
        
        # Sample the required percentage
        n_samples = max(1, int(len(all_files) * data_percentage))
        random.seed(42)
        sampled_files = random.sample(all_files, min(n_samples, len(all_files)))
        
        # Add sampled files to data
        for file_path in sampled_files:
            file_name = file_path.stem.strip()
            data.append({
                'video_id': f'HiDF_{label}_{folder_name}_{file_name}',
                'label': label,
                'path': str(file_path),
                'manipulation': manipulation,
                'dataset': 'HiDF',
                'media_type': media_type
            })
    
    df_result = pd.DataFrame(data)
    print(f"✓ HiDF: Loaded {len(df_result)} items ({data_percentage*100:.1f}%)")
    if len(df_result) > 0:
        print(f"  Real: {len(df_result[df_result['label'] == 'real'])}, Fake: {len(df_result[df_result['label'] == 'fake'])}")
        if 'media_type' in df_result.columns:
            vid_count = len(df_result[df_result['media_type'] == 'vid'])
            img_count = len(df_result[df_result['media_type'] == 'img'])
            print(f"  Videos: {vid_count}, Images: {img_count}")
    
    return df_result

In [8]:
# ============================================================================
# LOAD AND COMBINE ALL DATASETS
# ============================================================================

def load_all_datasets():
    """
    Load and combine all four datasets.
    Each dataset uses its own percentage from Config.
    
    Returns:
        DataFrame with combined metadata from all datasets
    """
    print(f"\n{'='*70}")
    print("LOADING DATASETS (per-dataset percentages)")
    print(f"{'='*70}")
    
    # Load each dataset (each uses its own config percentage)
    df_ffpp = load_ffpp_dataset()
    df_celebdf = load_celebdf_dataset()
    df_dfdc = load_dfdc_dataset()
    df_hidf = load_hidf_dataset()
    
    # Combine all datasets
    all_dfs = [df for df in [df_ffpp, df_celebdf, df_dfdc, df_hidf] if len(df) > 0]
    
    if not all_dfs:
        print("⚠ No datasets loaded!")
        return pd.DataFrame()
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    # Remove duplicates based on path
    combined_df = combined_df.drop_duplicates(subset=['path'])
    
    print(f"\n{'='*70}")
    print("COMBINED DATASET STATISTICS")
    print(f"{'='*70}")
    print(f"Total samples: {len(combined_df)}")
    print(f"Real: {len(combined_df[combined_df['label'] == 'real'])}, Fake: {len(combined_df[combined_df['label'] == 'fake'])}")
    print(f"\nSamples per dataset:")
    print(combined_df['dataset'].value_counts())
    print(f"\nManipulation types:")
    print(combined_df['manipulation'].value_counts())
    
    # Save combined metadata
    combined_df.to_csv(config.INTEGRATED_METADATA_PATH, index=False)
    print(f"\n✓ Metadata saved to: {config.INTEGRATED_METADATA_PATH}")
    
    return combined_df

# Load all datasets
integrated_metadata = load_all_datasets()


LOADING DATASETS (per-dataset percentages)
✓ FaceForensics++: Loaded 70 videos (1.0%)
  Real: 10, Fake: 60
✓ Celeb-DF: Loaded 25 videos (5.0%)
  Real: 12, Fake: 13
✓ DFDC: Loaded 120 videos (30.0%)
  Real: 19, Fake: 101
  ⚠ HiDF folder not found: Real-img
✓ HiDF: Loaded 3996 items (10.0%)
  Real: 436, Fake: 3560
  Videos: 872, Images: 3124

COMBINED DATASET STATISTICS
Total samples: 4211
Real: 477, Fake: 3734

Samples per dataset:
dataset
HiDF               3996
DFDC                120
FaceForensics++      70
Celeb-DF             25
Name: count, dtype: int64

Manipulation types:
manipulation
HiDF_FaceSwap        3560
Original              477
DFDC_DeepFake         101
Celeb-synthesis        13
Deepfakes              10
Face2Face              10
FaceShifter            10
NeuralTextures         10
FaceSwap               10
DeepFakeDetection      10
Name: count, dtype: int64

✓ Metadata saved to: integrated_nocuda.csv


## Section 3: Face Detection & Extraction

In [9]:
# ============================================================================
# FACE EXTRACTION (skips already-processed items automatically)
# ============================================================================

# Download OpenCV DNN face detector model files
FACE_PROTO = "deploy.prototxt"
FACE_MODEL = "res10_300x300_ssd_iter_140000_fp16.caffemodel"
PROTO_URL = "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt"
MODEL_URL = "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20180205_fp16/res10_300x300_ssd_iter_140000_fp16.caffemodel"

if not os.path.exists(FACE_PROTO):
    print("Downloading face detector prototxt...")
    urllib.request.urlretrieve(PROTO_URL, FACE_PROTO)
if not os.path.exists(FACE_MODEL):
    print("Downloading face detector model...")
    urllib.request.urlretrieve(MODEL_URL, FACE_MODEL)

face_net = cv2.dnn.readNetFromCaffe(FACE_PROTO, FACE_MODEL)
print("✓ OpenCV DNN face detector loaded")


# ============================================================================
# ROBUST VIDEO READING WITH FALLBACK BACKENDS
# cv2.VideoCapture often fails on SWAN CERN due to missing FFmpeg codecs.
# We try: cv2 → imageio → av (PyAV) in that order.
# ============================================================================

def _read_frames_cv2(file_path, num_frames):
    """Try reading video frames with OpenCV."""
    cap = cv2.VideoCapture(str(file_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None
    frame_indices = torch.linspace(0, total_frames - 1, num_frames).long()
    frames = []
    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx.item())
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames if frames else None


def _read_frames_imageio(file_path, num_frames):
    """Try reading video frames with imageio + imageio-ffmpeg."""
    try:
        import imageio.v3 as iio
        frames_all = iio.imread(str(file_path), plugin="pyav")
        total = len(frames_all)
        if total == 0:
            return None
        indices = torch.linspace(0, total - 1, min(num_frames, total)).long().tolist()
        return [frames_all[i] for i in indices]
    except Exception:
        pass
    try:
        import imageio
        reader = imageio.get_reader(str(file_path), format='ffmpeg')
        total = reader.count_frames()
        if total == 0:
            reader.close()
            return None
        indices = set(torch.linspace(0, total - 1, min(num_frames, total)).long().tolist())
        frames = []
        for i, frame in enumerate(reader):
            if i in indices:
                frames.append(frame)  # Already RGB
            if len(frames) >= num_frames:
                break
        reader.close()
        return frames if frames else None
    except Exception:
        return None


def _read_frames_av(file_path, num_frames):
    """Try reading video frames with PyAV."""
    try:
        import av
        container = av.open(str(file_path))
        stream = container.streams.video[0]
        total = stream.frames
        if total == 0:
            # Try counting manually
            total = sum(1 for _ in container.decode(video=0))
            container.close()
            container = av.open(str(file_path))
            if total == 0:
                container.close()
                return None
        indices = set(torch.linspace(0, max(total - 1, 0), min(num_frames, max(total, 1))).long().tolist())
        frames = []
        for i, frame in enumerate(container.decode(video=0)):
            if i in indices:
                frames.append(frame.to_ndarray(format='rgb24'))
            if len(frames) >= num_frames:
                break
        container.close()
        return frames if frames else None
    except Exception:
        return None


# Detect which video backend works (tested once at startup)
_VIDEO_BACKEND = None

def _detect_video_backend(test_path=None):
    """Test which video reading backend works on this system."""
    global _VIDEO_BACKEND
    if _VIDEO_BACKEND is not None:
        return _VIDEO_BACKEND

    # If no test video provided, just check module availability
    backends = []
    
    # Check cv2 video support
    try:
        cap = cv2.VideoCapture()
        backends.append('cv2')
    except Exception:
        pass
    
    try:
        import imageio
        backends.append('imageio')
    except ImportError:
        pass
    
    try:
        import av
        backends.append('av')
    except ImportError:
        pass
    
    print(f"  Available video backends: {backends}")
    
    if test_path:
        # Actually test with a real video
        for backend_name, read_fn in [('cv2', _read_frames_cv2), ('imageio', _read_frames_imageio), ('av', _read_frames_av)]:
            if backend_name not in backends:
                continue
            result = read_fn(test_path, 2)
            if result and len(result) > 0:
                print(f"  ✓ Working video backend: {backend_name}")
                _VIDEO_BACKEND = backend_name
                return _VIDEO_BACKEND
            else:
                print(f"  ✗ {backend_name} failed to read test video")
        
        print("  ⚠ No video backend could read the test video!")
        _VIDEO_BACKEND = 'none'
    else:
        _VIDEO_BACKEND = backends[0] if backends else 'none'
    
    return _VIDEO_BACKEND


def read_video_frames(file_path, num_frames=20):
    """
    Read frames from a video file using the best available backend.
    Falls back through cv2 → imageio → av until one works.
    
    Returns:
        List of RGB numpy arrays, or empty list if all backends fail.
    """
    for read_fn in [_read_frames_cv2, _read_frames_imageio, _read_frames_av]:
        result = read_fn(file_path, num_frames)
        if result and len(result) > 0:
            return result
    return []


def detect_and_crop_face(image_rgb, image_size=299, confidence_threshold=0.5):
    """Detect and crop the largest face using OpenCV DNN. Returns PIL Image or None."""
    h, w = image_rgb.shape[:2]
    blob = cv2.dnn.blobFromImage(image_rgb, 1.0, (300, 300), (104.0, 177.0, 123.0))
    face_net.setInput(blob)
    detections = face_net.forward()

    best_face = None
    best_conf = confidence_threshold

    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > best_conf:
            x1 = max(0, int(detections[0, 0, i, 3] * w))
            y1 = max(0, int(detections[0, 0, i, 4] * h))
            x2 = min(w, int(detections[0, 0, i, 5] * w))
            y2 = min(h, int(detections[0, 0, i, 6] * h))
            # Add margin
            margin = int(0.15 * max(x2 - x1, y2 - y1))
            x1 = max(0, x1 - margin)
            y1 = max(0, y1 - margin)
            x2 = min(w, x2 + margin)
            y2 = min(h, y2 + margin)
            best_face = (x1, y1, x2, y2)
            best_conf = conf

    if best_face is None:
        return None

    x1, y1, x2, y2 = best_face
    face_crop = image_rgb[y1:y2, x1:x2]
    face_pil = Image.fromarray(face_crop).resize((image_size, image_size), Image.LANCZOS)
    return face_pil


def extract_faces_integrated(metadata_df, output_dir, frames_per_video=20, image_size=299):
    """
    Extract faces from videos/images using OpenCV DNN face detector.
    Handles both video and image inputs.
    Uses fallback video backends (cv2 → imageio → av) for SWAN CERN compatibility.
    AUTOMATICALLY SKIPS already-processed items based on output directory existence.

    Args:
        metadata_df: DataFrame with video/image metadata
        output_dir: Directory to save extracted faces
        frames_per_video: Number of frames to sample from videos
        image_size: Output face image size
    """
    print(f"\n{'='*70}")
    print("FACE EXTRACTION WITH OPENCV DNN")
    print(f"{'='*70}")

    # ---- Diagnostic: find a test video and check which backend works ----
    test_video = None
    for _, row in metadata_df.iterrows():
        ext = Path(row['path']).suffix.lower()
        if ext in ['.mp4', '.avi', '.mov', '.mkv']:
            if Path(row['path']).exists():
                test_video = row['path']
                break

    if test_video:
        print(f"\n  Testing video reading with: {Path(test_video).name}")
        backend = _detect_video_backend(test_video)
        if backend == 'none':
            print("  ⚠ WARNING: No video backend works! Video files will be skipped.")
            print("  Try: pip install imageio imageio-ffmpeg av")
    else:
        print("  No video files found in metadata, skipping video backend test.")
        _detect_video_backend()

    processed = 0
    skipped = 0
    errors = 0
    video_errors = 0

    for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Extracting faces"):
        file_path = row['path']
        video_id = row['video_id']
        label = row['label']
        media_type = row.get('media_type', 'vid')  # Default to video

        # Determine media type from file extension if not specified
        ext = Path(file_path).suffix.lower()
        if ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            media_type = 'img'
        elif ext in ['.mp4', '.avi', '.mov', '.mkv']:
            media_type = 'vid'

        # Sanitize video_id: strip whitespace to avoid path errors
        video_id = str(video_id).strip()
        video_output_dir = output_dir / label / video_id

        # Skip if already processed (key logic for re-runs)
        if video_output_dir.exists():
            existing_files = list(video_output_dir.glob('*.png'))
            min_expected = 1 if media_type == 'img' else max(1, frames_per_video // 4)
            if len(existing_files) >= min_expected:
                skipped += 1
                continue

        video_output_dir.mkdir(parents=True, exist_ok=True)

        try:
            if media_type == 'img':
                # Handle image
                img = cv2.imread(file_path)
                if img is None:
                    errors += 1
                    continue

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                face_pil = detect_and_crop_face(img_rgb, image_size)

                if face_pil is not None:
                    save_path = video_output_dir / 'frame_0.png'
                    face_pil.save(str(save_path))
                    processed += 1
                else:
                    errors += 1

            else:
                # Handle video using robust multi-backend reader
                frames_rgb = read_video_frames(file_path, frames_per_video)

                if not frames_rgb:
                    video_errors += 1
                    errors += 1
                    if video_errors <= 3:
                        print(f"\n  ⚠ Video read failed: {Path(file_path).name}")
                    elif video_errors == 4:
                        print(f"\n  ⚠ Suppressing further video error messages...")
                    continue

                saved_count = 0
                for frame_rgb in frames_rgb:
                    face_pil = detect_and_crop_face(frame_rgb, image_size)

                    if face_pil is not None:
                        save_path = video_output_dir / f'frame_{saved_count}.png'
                        face_pil.save(str(save_path))
                        saved_count += 1

                    if saved_count >= frames_per_video:
                        break

                if saved_count > 0:
                    processed += 1
                else:
                    errors += 1

        except Exception as e:
            errors += 1

    print(f"\n{'='*70}")
    print(f"Face extraction complete!")
    print(f"Processed: {processed}, Skipped (already done): {skipped}, Errors: {errors}")
    if video_errors > 0:
        print(f"  ⚠ Video read failures: {video_errors} (try: pip install imageio imageio-ffmpeg av)")
    print(f"{'='*70}")

# Run face extraction
extract_faces_integrated(
    integrated_metadata, 
    config.PREPROCESSED_DIR,
    config.FRAMES_PER_VIDEO, 
    config.FACE_SIZE)

✓ OpenCV DNN face detector loaded

FACE EXTRACTION WITH OPENCV DNN

  Testing video reading with: 521.mp4
  Available video backends: ['cv2', 'imageio', 'av']
  ✗ cv2 failed to read test video
  ✓ Working video backend: imageio


Extracting faces: 100%|██████████| 4211/4211 [02:15<00:00, 31.06it/s] 


Face extraction complete!
Processed: 13, Skipped (already done): 4166, Errors: 32


## Section 4: Feature Extractors

In [10]:
# ============================================================================
# FEATURE EXTRACTORS
# ============================================================================

class SpatialExtractor(nn.Module):
    """XceptionNet-based spatial feature extractor (2048-dim)."""

    def __init__(self):
        super().__init__()
        self.model = timm.create_model('xception', pretrained=True, num_classes=0)
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    @torch.no_grad()
    def forward(self, img_path):
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(img).unsqueeze(0).to(device)
        return self.model(img_tensor).squeeze().cpu().numpy()


class FrequencyExtractor:
    """FFT-based frequency feature extractor (128-dim azimuthal power spectrum).

    Instead of 4 scalar statistics, computes the radial (azimuthally-averaged)
    power spectrum profile, preserving spectral shape information that reveals
    GAN artifacts and compression patterns invisible in summary statistics.
    """

    FREQ_DIM = 128

    def __init__(self):
        self.transform = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.Grayscale(),
            transforms.ToTensor(),
        ])

    def extract(self, img_path):
        img = Image.open(img_path)
        img_tensor = self.transform(img).squeeze(0)  # (299, 299)
        h, w = img_tensor.shape

        # Hann window to reduce spectral leakage
        window = torch.hann_window(h).unsqueeze(1) * torch.hann_window(w).unsqueeze(0)
        img_windowed = img_tensor * window

        # 2D FFT -> power spectrum
        f_transform = torch.fft.fft2(img_windowed)
        f_shift = torch.fft.fftshift(f_transform)
        power_spectrum = (torch.abs(f_shift) ** 2).numpy()

        # Azimuthal average (radial power spectrum profile)
        cy, cx = h // 2, w // 2
        Y, X = np.ogrid[:h, :w]
        radius = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2).astype(int)
        max_radius = min(cy, cx)

        radial_profile = np.zeros(max_radius)
        for r in range(max_radius):
            mask = radius == r
            if mask.any():
                radial_profile[r] = power_spectrum[mask].mean()

        # Log-scale (avoid log(0))
        radial_profile = np.log1p(radial_profile)

        # Resample to fixed FREQ_DIM via linear interpolation
        x_old = np.linspace(0, 1, len(radial_profile))
        x_new = np.linspace(0, 1, self.FREQ_DIM)
        resampled = np.interp(x_new, x_old, radial_profile)

        return resampled.astype(np.float32)


class SemanticExtractor(nn.Module):
    """
    CLIP ViT-B/32 semantic feature extractor (768-dim).

    NOTE: DINOv2 requires xFormers which has compatibility issues.
    Using CLIP ViT-B/32 via timm instead - reliable on all platforms.
    """

    def __init__(self):
        super().__init__()
        self.model = timm.create_model('vit_base_patch32_clip_224.openai', pretrained=True, num_classes=0)
        self.model.eval()

        # Get proper transforms from timm
        data_config = timm.data.resolve_model_data_config(self.model)
        self.transform = timm.data.create_transform(**data_config, is_training=False)

    @torch.no_grad()
    def forward(self, img_path):
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(img).unsqueeze(0).to(device)
        return self.model(img_tensor).squeeze().cpu().numpy()

print("Feature extractors defined:")
print(f"  - SpatialExtractor (Xception): {config.SPATIAL_DIM}-dim")
print(f"  - FrequencyExtractor (azimuthal power spectrum): {config.FREQ_DIM}-dim")
print(f"  - SemanticExtractor (CLIP ViT-B/32): {config.SEMANTIC_DIM}-dim")

In [11]:
# ============================================================================
# EXTRACT ALL FEATURES
# ============================================================================

def extract_all_features(preprocessed_dir, output_dir, force_semantic=False):
    """
    Extract features from all three domains.
    Skips already-extracted features for ALL domains (spatial, frequency, semantic).
    
    Args:
        preprocessed_dir: Path to preprocessed face images
        output_dir: Path to save extracted features
        force_semantic: If True, re-extract ALL semantic features (use ONLY when
                        switching semantic models to avoid mixed feature spaces).
                        Set to False for normal runs to skip existing features.
    """
    
    # Collect all image paths
    image_paths = []
    for label in ['real', 'fake']:
        label_dir = preprocessed_dir / label
        if label_dir.exists():
            for video_dir in label_dir.iterdir():
                if video_dir.is_dir():
                    for img_path in video_dir.glob('*.png'):
                        image_paths.append(img_path)
    
    print(f"\nFound {len(image_paths)} images to process")
    
    if len(image_paths) == 0:
        print("No images found. Run face extraction first.")
        return
    
    # If force_semantic, delete old semantic features to avoid mixing models
    if force_semantic:
        import shutil
        semantic_dir = output_dir / 'semantic'
        if semantic_dir.exists():
            old_count = sum(1 for _ in semantic_dir.rglob('*.npy'))
            shutil.rmtree(semantic_dir)
            semantic_dir.mkdir(parents=True, exist_ok=True)
            print(f"Cleared {old_count} old semantic features (force re-extraction)")
    
    # Extract each domain sequentially, freeing memory between domains
    domains_config = [
        ('spatial', lambda: SpatialExtractor().to(device), lambda e, p: e(p)),
        ('frequency', lambda: FrequencyExtractor(), lambda e, p: e.extract(p)),
        ('semantic', lambda: SemanticExtractor().to(device), lambda e, p: e(p)),
    ]
    
    for domain, init_fn, extract_fn in domains_config:
        print(f"\n{'='*70}")
        print(f"EXTRACTING {domain.upper()} FEATURES")
        print(f"{'='*70}")
        
        # Initialize extractor fresh for each domain
        extractor = init_fn()
        
        processed = 0
        skipped = 0
        errors = 0
        
        for img_path in tqdm(image_paths, desc=f"Extracting {domain}"):
            relative_path = img_path.relative_to(preprocessed_dir)
            feature_path = output_dir / domain / relative_path.with_suffix('.npy')
            
            # Skip if feature already exists (applies to ALL domains)
            if feature_path.exists():
                skipped += 1
                continue
            
            feature_path.parent.mkdir(parents=True, exist_ok=True)
            
            try:
                features = extract_fn(extractor, img_path)
                np.save(feature_path, features)
                processed += 1
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  Error on {img_path.name}: {e}")
        
        print(f"  {domain.capitalize()} complete: Processed={processed}, Skipped={skipped}, Errors={errors}")
        
        # Free memory before next domain
        del extractor

# Run feature extraction
# Set force_semantic=True ONLY when switching semantic models (e.g., DINOv2 → CLIP)
# Set force_semantic=False for normal runs (skips existing features like spatial/frequency)
extract_all_features(config.PREPROCESSED_DIR, config.FEATURES_DIR, force_semantic=False)


Found 51818 images to process

EXTRACTING SPATIAL FEATURES


Extracting spatial: 100%|██████████| 51818/51818 [06:23<00:00, 135.06it/s]


  Spatial complete: Processed=0, Skipped=51818, Errors=0

EXTRACTING FREQUENCY FEATURES


Extracting frequency: 100%|██████████| 51818/51818 [06:39<00:00, 129.57it/s] 


  Frequency complete: Processed=0, Skipped=51818, Errors=0

EXTRACTING SEMANTIC FEATURES


Extracting semantic: 100%|██████████| 51818/51818 [06:38<00:00, 129.90it/s] 

  Semantic complete: Processed=0, Skipped=51818, Errors=0


In [12]:
# ============================================================================
# PIPELINE TRACKING CSV - Counts per dataset at each stage
# ============================================================================

def generate_pipeline_tracking_csv(metadata_df, preprocessed_dir, features_dir, save_path=None):
    """
    Generate a CSV that tracks how many items from each dataset 
    go through each pipeline stage: loading, preprocessing, feature extraction, training.
    
    Shows counts broken down by dataset, label, and media type.
    """
    import time  # DEBUG: for timing each stage
    
    save_path = save_path or config.BASE_DIR / 'pipeline_tracking.csv'
    
    prefix_to_dataset = {
        'FFPP_': 'FaceForensics++',
        'CelebDF_': 'Celeb-DF',
        'DFDC_': 'DFDC',
        'HiDF_': 'HiDF'
    }
    
    tracking_rows = []
    
    # ========== DEBUG: Stage 1 Start ==========
    print("[DEBUG] >>> STAGE 1: Loaded metadata - START")
    stage1_start = time.time()
    
    # --- Stage 1: Loaded metadata ---
    if metadata_df is not None and len(metadata_df) > 0:
        print(f"[DEBUG] Stage 1: metadata has {len(metadata_df)} rows, {metadata_df['dataset'].nunique()} datasets")
        for ds in metadata_df['dataset'].unique():
            ds_df = metadata_df[metadata_df['dataset'] == ds]
            real = len(ds_df[ds_df['label'] == 'real'])
            fake = len(ds_df[ds_df['label'] == 'fake'])
            
            # Count videos vs images - VECTORIZED instead of iterrows (was slow!)
            extensions = ds_df['path'].apply(lambda p: Path(p).suffix.lower())
            img_mask = extensions.isin(['.jpg', '.jpeg', '.png', '.bmp'])
            img_count = int(img_mask.sum())
            vid_count = len(ds_df) - img_count
            
            print(f"[DEBUG]   Dataset '{ds}': total={len(ds_df)}, real={real}, fake={fake}, vid={vid_count}, img={img_count}")
            
            tracking_rows.append({
                'Stage': '1_Loaded',
                'Dataset': ds,
                'Total': len(ds_df),
                'Real': real,
                'Fake': fake,
                'Videos': vid_count,
                'Images': img_count
            })
    else:
        print("[DEBUG] Stage 1: metadata_df is None or empty!")
    
    stage1_time = time.time() - stage1_start
    print(f"[DEBUG] <<< STAGE 1 COMPLETE in {stage1_time:.2f}s, tracking_rows so far: {len(tracking_rows)}")
    
    # ========== DEBUG: Stage 2 Start ==========
    print(f"\n[DEBUG] >>> STAGE 2: Preprocessed faces - START")
    print(f"[DEBUG] Stage 2: preprocessed_dir = {preprocessed_dir}")
    stage2_start = time.time()
    stage2_count = 0
    
    # --- Stage 2: Preprocessed faces ---
    for label in ['real', 'fake']:
        label_dir = preprocessed_dir / label
        if not label_dir.exists():
            print(f"[DEBUG]   Stage 2: {label_dir} does NOT exist, skipping")
            continue
        
        # DEBUG: Count total dirs first so we can show progress
        all_dirs = [d for d in label_dir.iterdir() if d.is_dir()]
        total_dirs = len(all_dirs)
        print(f"[DEBUG]   Stage 2 '{label}': found {total_dirs} directories to scan")
        
        for i, video_dir in enumerate(all_dirs):
            # DEBUG: Progress every 500 directories
            if (i + 1) % 500 == 0 or i == 0:
                elapsed = time.time() - stage2_start
                print(f"[DEBUG]   Stage 2 '{label}': scanning dir {i+1}/{total_dirs} ({elapsed:.1f}s elapsed)")
            
            video_id = video_dir.name
            # OPTIMIZATION: Use os.listdir + filter instead of glob (faster on Windows)
            try:
                num_frames = sum(1 for f in os.listdir(video_dir) if f.endswith('.png'))
            except OSError as e:
                print(f"[DEBUG]   Stage 2 ERROR reading {video_dir}: {e}")
                continue
            if num_frames == 0:
                continue
            
            ds_name = None
            for prefix, name in prefix_to_dataset.items():
                if video_id.startswith(prefix):
                    ds_name = name
                    break
            if ds_name is None:
                ds_name = 'Unknown'
            
            tracking_rows.append({
                'Stage': '2_Preprocessed',
                'Dataset': ds_name,
                'Label': label,
                'Videos_or_Items': 1,
                'Face_Images': num_frames
            })
            stage2_count += 1
        
        print(f"[DEBUG]   Stage 2 '{label}': DONE scanning {total_dirs} dirs")
    
    stage2_time = time.time() - stage2_start
    print(f"[DEBUG] <<< STAGE 2 COMPLETE in {stage2_time:.2f}s, added {stage2_count} rows, tracking_rows total: {len(tracking_rows)}")
    
    # ========== DEBUG: Stage 3 Start ==========
    print(f"\n[DEBUG] >>> STAGE 3: Feature extraction - START")
    print(f"[DEBUG] Stage 3: features_dir = {features_dir}")
    stage3_start = time.time()
    stage3_count = 0
    
    # --- Stage 3: Feature extraction ---
    for domain in ['spatial', 'frequency', 'semantic']:
        domain_start = time.time()
        print(f"[DEBUG]   Stage 3 domain '{domain}': starting...")
        
        for label in ['real', 'fake']:
            domain_label_dir = features_dir / domain / label
            if not domain_label_dir.exists():
                print(f"[DEBUG]     {domain}/{label}: directory does NOT exist, skipping")
                continue
            
            # DEBUG: Count total dirs first 
            all_dirs = [d for d in domain_label_dir.iterdir() if d.is_dir()]
            total_dirs = len(all_dirs)
            print(f"[DEBUG]     {domain}/{label}: found {total_dirs} directories to scan")
            
            for i, video_dir in enumerate(all_dirs):
                # DEBUG: Progress every 1000 directories
                if (i + 1) % 1000 == 0 or i == 0:
                    elapsed = time.time() - domain_start
                    print(f"[DEBUG]     {domain}/{label}: scanning dir {i+1}/{total_dirs} ({elapsed:.1f}s elapsed)")
                
                video_id = video_dir.name
                # OPTIMIZATION: Use os.listdir + filter instead of glob (faster on Windows)
                try:
                    num_features = sum(1 for f in os.listdir(video_dir) if f.endswith('.npy'))
                except OSError as e:
                    print(f"[DEBUG]     Stage 3 ERROR reading {video_dir}: {e}")
                    continue
                if num_features == 0:
                    continue
                
                ds_name = None
                for prefix, name in prefix_to_dataset.items():
                    if video_id.startswith(prefix):
                        ds_name = name
                        break
                if ds_name is None:
                    ds_name = 'Unknown'
                
                tracking_rows.append({
                    'Stage': f'3_Features_{domain}',
                    'Dataset': ds_name,
                    'Label': label,
                    'Videos_or_Items': 1,
                    'Feature_Files': num_features
                })
                stage3_count += 1
            
            print(f"[DEBUG]     {domain}/{label}: DONE scanning {total_dirs} dirs")
        
        domain_time = time.time() - domain_start
        print(f"[DEBUG]   Stage 3 domain '{domain}': COMPLETE in {domain_time:.2f}s")
    
    stage3_time = time.time() - stage3_start
    print(f"[DEBUG] <<< STAGE 3 COMPLETE in {stage3_time:.2f}s, added {stage3_count} rows, tracking_rows total: {len(tracking_rows)}")
    
    # ========== DEBUG: Aggregation Start ==========
    print(f"\n[DEBUG] >>> AGGREGATION: Building summary from {len(tracking_rows)} raw rows")
    agg_start = time.time()
    
    # Build the final tracking dataframe
    raw_df = pd.DataFrame(tracking_rows)
    print(f"[DEBUG] raw_df shape: {raw_df.shape}, columns: {list(raw_df.columns)}")
    
    # Aggregate into a clean summary
    summary_rows = []
    
    # Stage 1 summary (already aggregated)
    stage1 = raw_df[raw_df['Stage'] == '1_Loaded']
    print(f"[DEBUG] Stage 1 rows in raw_df: {len(stage1)}")
    for _, row in stage1.iterrows():
        summary_rows.append(row.to_dict())
    
    # Stage 2 summary (aggregate)
    stage2 = raw_df[raw_df['Stage'] == '2_Preprocessed']
    print(f"[DEBUG] Stage 2 rows in raw_df: {len(stage2)}")
    if len(stage2) > 0:
        for ds in stage2['Dataset'].unique():
            ds_data = stage2[stage2['Dataset'] == ds]
            for label in ['real', 'fake']:
                label_data = ds_data[ds_data['Label'] == label]
                if len(label_data) > 0:
                    summary_rows.append({
                        'Stage': '2_Preprocessed',
                        'Dataset': ds,
                        'Label': label,
                        'Items_Count': int(label_data['Videos_or_Items'].sum()),
                        'Total_Face_Images': int(label_data['Face_Images'].sum())
                    })
    
    # Stage 3 summary (aggregate per domain)
    for domain in ['spatial', 'frequency', 'semantic']:
        stage3 = raw_df[raw_df['Stage'] == f'3_Features_{domain}']
        print(f"[DEBUG] Stage 3 '{domain}' rows in raw_df: {len(stage3)}")
        if len(stage3) > 0:
            for ds in stage3['Dataset'].unique():
                ds_data = stage3[stage3['Dataset'] == ds]
                for label in ['real', 'fake']:
                    label_data = ds_data[ds_data['Label'] == label]
                    if len(label_data) > 0:
                        summary_rows.append({
                            'Stage': f'3_Features_{domain}',
                            'Dataset': ds,
                            'Label': label,
                            'Items_Count': int(label_data['Videos_or_Items'].sum()),
                            'Total_Feature_Files': int(label_data['Feature_Files'].sum())
                        })
    
    agg_time = time.time() - agg_start
    print(f"[DEBUG] <<< AGGREGATION COMPLETE in {agg_time:.2f}s, summary has {len(summary_rows)} rows")
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(save_path, index=False)
    print(f"[DEBUG] CSV saved to {save_path}")
    
    # Print a clean overview
    print(f"\n{'='*70}")
    print("PIPELINE TRACKING SUMMARY")
    print(f"{'='*70}")
    
    # Stage 1
    print("\n--- Stage 1: Loaded Metadata ---")
    s1 = summary_df[summary_df['Stage'] == '1_Loaded']
    if len(s1) > 0:
        for _, r in s1.iterrows():
            print(f"  {r['Dataset']:20s}: Total={int(r.get('Total',0)):5d}, Real={int(r.get('Real',0)):5d}, Fake={int(r.get('Fake',0)):5d}, Videos={int(r.get('Videos',0)):5d}, Images={int(r.get('Images',0)):5d}")
    
    # Stage 2
    print("\n--- Stage 2: Preprocessed Faces ---")
    s2 = summary_df[summary_df['Stage'] == '2_Preprocessed']
    if len(s2) > 0:
        for ds in s2['Dataset'].unique():
            ds_data = s2[s2['Dataset'] == ds]
            items = int(ds_data['Items_Count'].sum())
            faces = int(ds_data['Total_Face_Images'].sum())
            real_items = int(ds_data[ds_data['Label']=='real']['Items_Count'].sum()) if 'real' in ds_data['Label'].values else 0
            fake_items = int(ds_data[ds_data['Label']=='fake']['Items_Count'].sum()) if 'fake' in ds_data['Label'].values else 0
            print(f"  {ds:20s}: Items={items:5d} (Real={real_items}, Fake={fake_items}), Total_Faces={faces:6d}")
    
    # Stage 3
    print("\n--- Stage 3: Feature Extraction ---")
    for domain in ['spatial', 'frequency', 'semantic']:
        s3 = summary_df[summary_df['Stage'] == f'3_Features_{domain}']
        if len(s3) > 0:
            total_features = int(s3['Total_Feature_Files'].sum()) if 'Total_Feature_Files' in s3.columns else 0
            print(f"  {domain.upper():12s}: Total={total_features:6d}", end="")
            for ds in s3['Dataset'].unique():
                ds_feats = int(s3[s3['Dataset']==ds]['Total_Feature_Files'].sum())
                print(f"  | {ds}={ds_feats}", end="")
            print()
    
    print(f"\nTracking CSV saved to: {save_path}")
    total_time = stage1_time + stage2_time + stage3_time + agg_time
    print(f"[DEBUG] TOTAL pipeline tracking time: {total_time:.2f}s")
    return summary_df

# Generate tracking CSV
print("[DEBUG] Calling generate_pipeline_tracking_csv...")
tracking_df = generate_pipeline_tracking_csv(
    integrated_metadata, 
    config.PREPROCESSED_DIR, 
    config.FEATURES_DIR
)
print("[DEBUG] generate_pipeline_tracking_csv FINISHED")

[DEBUG] Calling generate_pipeline_tracking_csv...
[DEBUG] >>> STAGE 1: Loaded metadata - START
[DEBUG] Stage 1: metadata has 4211 rows, 4 datasets
[DEBUG]   Dataset 'FaceForensics++': total=70, real=10, fake=60, vid=70, img=0
[DEBUG]   Dataset 'Celeb-DF': total=25, real=12, fake=13, vid=25, img=0
[DEBUG]   Dataset 'DFDC': total=120, real=19, fake=101, vid=120, img=0
[DEBUG]   Dataset 'HiDF': total=3996, real=436, fake=3560, vid=872, img=3124
[DEBUG] <<< STAGE 1 COMPLETE in 0.04s, tracking_rows so far: 4

[DEBUG] >>> STAGE 2: Preprocessed faces - START
[DEBUG] Stage 2: preprocessed_dir = preprocessed_faces_integrated
[DEBUG]   Stage 2 'real': found 5616 directories to scan
[DEBUG]   Stage 2 'real': scanning dir 1/5616 (7.0s elapsed)
[DEBUG]   Stage 2 'real': scanning dir 500/5616 (12.8s elapsed)
[DEBUG]   Stage 2 'real': scanning dir 1000/5616 (17.9s elapsed)
[DEBUG]   Stage 2 'real': scanning dir 1500/5616 (23.4s elapsed)
[DEBUG]   Stage 2 'real': scanning dir 2000/5616 (29.3s elapsed)

[DEBUG]     spatial/fake: scanning dir 6000/31827 (175.3s elapsed)
[DEBUG]     spatial/fake: scanning dir 7000/31827 (189.9s elapsed)
[DEBUG]     spatial/fake: scanning dir 8000/31827 (202.3s elapsed)
[DEBUG]     spatial/fake: scanning dir 9000/31827 (214.4s elapsed)
[DEBUG]     spatial/fake: scanning dir 10000/31827 (228.0s elapsed)
[DEBUG]     spatial/fake: scanning dir 11000/31827 (240.7s elapsed)
[DEBUG]     spatial/fake: scanning dir 12000/31827 (253.0s elapsed)
[DEBUG]     spatial/fake: scanning dir 13000/31827 (264.9s elapsed)
[DEBUG]     spatial/fake: scanning dir 14000/31827 (276.5s elapsed)
[DEBUG]     spatial/fake: scanning dir 15000/31827 (288.6s elapsed)
[DEBUG]     spatial/fake: scanning dir 16000/31827 (300.8s elapsed)
[DEBUG]     spatial/fake: scanning dir 17000/31827 (313.5s elapsed)
[DEBUG]     spatial/fake: scanning dir 18000/31827 (325.3s elapsed)
[DEBUG]     Stage 3 ERROR reading extracted_features_integrated/spatial/fake/HiDF_fake_Fake-img_f20154_c19753: [Errno 1]

[DEBUG] generate_pipeline_tracking_csv FINISHED


## Save Preprocessing Manifest

Scan all extracted features and save a manifest CSV listing every sample
with complete features across all 3 domains. The training notebook loads
this manifest instead of re-scanning the filesystem.

In [ ]:
# ============================================================================
# SAVE PREPROCESSING MANIFEST
# ============================================================================
# Scans the features directory and saves a CSV listing every sample that has
# complete features across all 3 domains (spatial, frequency, semantic).
# The training notebook loads this manifest instead of re-scanning the filesystem.

def save_preprocessing_manifest(features_dir, save_path):
    """Scan features directory and save manifest of complete samples."""
    prefix_to_dataset = {
        'FFPP_': 'FaceForensics++',
        'CelebDF_': 'Celeb-DF',
        'DFDC_': 'DFDC',
        'HiDF_': 'HiDF'
    }
    required_domains = ['spatial', 'frequency', 'semantic']

    rows = []
    for label in ['real', 'fake']:
        spatial_label_dir = features_dir / 'spatial' / label
        if not spatial_label_dir.exists():
            continue

        video_dirs = [d for d in spatial_label_dir.iterdir() if d.is_dir()]
        print(f"  Scanning {label}: {len(video_dirs)} video directories...")

        for video_dir in tqdm(video_dirs, desc=f"Building manifest ({label})"):
            video_id = video_dir.name

            # Determine dataset from prefix
            dataset_name = None
            for prefix, ds_name in prefix_to_dataset.items():
                if video_id.startswith(prefix):
                    dataset_name = ds_name
                    break
            if dataset_name is None:
                continue

            # Check each frame for complete features across all domains
            for feature_file in video_dir.glob('*.npy'):
                frame_name = feature_file.stem
                all_exist = True
                for domain in required_domains:
                    path = features_dir / domain / label / video_id / f'{frame_name}.npy'
                    if not path.exists():
                        all_exist = False
                        break
                if all_exist:
                    rows.append({
                        'video_id': video_id,
                        'label': label,
                        'dataset': dataset_name,
                        'frame_name': frame_name
                    })

    manifest_df = pd.DataFrame(rows)
    manifest_df.to_csv(save_path, index=False)

    print(f"\n{'='*70}")
    print("PREPROCESSING MANIFEST SAVED")
    print(f"{'='*70}")
    print(f"Total samples with complete features: {len(manifest_df)}")
    if len(manifest_df) > 0:
        print(f"\nBy label:")
        for lbl, count in manifest_df['label'].value_counts().items():
            print(f"  {lbl}: {count}")
        print(f"\nBy dataset:")
        for ds, count in manifest_df['dataset'].value_counts().items():
            print(f"  {ds}: {count}")
    print(f"\nSaved to: {save_path}")
    print(f"\nNext step: Open train_sections5_end.ipynb to train the model.")
    return manifest_df

manifest_path = config.BASE_DIR / 'preprocessing_manifest.csv'
manifest_df = save_preprocessing_manifest(config.FEATURES_DIR, manifest_path)